# 8 · Desktop screens, and a model told how the stream was made

Two things notebook 4 found standing between the model and a better score, and neither of
them is the network:

- **It had never seen a desktop.** The patch set was photos and YouTube video; the
  validation set had no text, no UI and no code in it, so it could not even say how the
  model does on what a remote desktop mostly shows. `datasets/screens.py` now draws
  desktop screens - code editors, documents, terminals, spreadsheets, chat, file
  explorers, settings, in real fonts at 100-200% display scale - and
  `0_data_preprocess.ipynb` puts them in both splits.
- **It had to guess how each frame was made.** Its gain depends on the downscale filter
  (+1.8 dB on bilinear, +0.6 on area) and on the bitrate, and it was told neither. The
  client knows the bitrate exactly - it asked for the bandwidth, the height and the frame
  rate itself - and it would know the rest if the host said which encoder line it runs.

So this notebook trains notebook 7's body four ways on the same data: blind (three seeds,
which is the ruler), told the bitrate, and told the bitrate and the host's encoder path;
and it measures notebook 4's model, trained before there were screens, on the same set.

Steps:

1. **Data** - the patch set with screens, how a patch is grouped, and what a model is told.
2. **Model** - notebook 7's body, and the same body with a condition.
3. **Speed** - the shipped graph, which the condition does not change.
4. **Train** - blind three times, then told the bitrate, then told the path too.
5. **Results** - overall, on screens against photos and video, by display scale, bitrate
   and filter, and on what the host sends today.
6. **Save** and **Export** - the best, at the client's tile.

Needs: `datasets/screens.py`, then `0_data_preprocess.ipynb`. Results: `summary.md`,
"Desktop screens and a known stream". Writes `checkpoints/upscale_screens.pt`, and each
run's training to `checkpoints/screens_runs/`.

In [ ]:
"""Settings."""

import copy
import json

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Subset

from helpers import degrade, metrics, models, patches, sources, speed, tiles, training, webexport
from helpers.training import DATA_DIR, DEVICE

CHECKPOINT = training.CHECKPOINT_DIR / "upscale_screens.pt"   # the winner, as it ships
RUN_DIR = training.CHECKPOINT_DIR / "screens_runs"            # each run's training, saved every epoch
BEFORE = training.CHECKPOINT_DIR / "upscale_strategies.pt"    # notebook 4's winner, trained without screens

WIDTH, BLOCKS, STEM = 24, 2, 3   # notebook 7's body, the best of notebook 4
MAX_EPOCHS = 500                 # a safety limit; each training stops by itself once it has converged
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
SEEDS = (0, 1, 2)                # the blind model trained three times: how far luck alone moves it
TOLD = {                         # what a model is told about its frames: parts of degrade.CONDITIONS
    "told the bitrate": ("bpp",),
    "told the bitrate and path": ("bpp", "path"),
}
# what the host sends today, on every ffmpeg line: swscale's bicubic, libx264 with no loop filter
HOST_TODAY = {"filter": "bicubic", "deblock": False}
BAKE_BPP = (0.2, degrade.STREAM_BPP, 1.0)   # bitrates a told model is exported at, one graph each

STEP_WIDTH, STEP_HEIGHT = 320, 180       # LR pixels one client tile adds to the picture (notebook 5)
FRAME_WIDTH, FRAME_HEIGHT = 1920, 1080   # the whole-frame graph
NAME = "screens_24x2"
ESTIMATE_ON = {"Iris Xe": "Intel Iris Xe, 96 EU (laptop iGPU)", "GTX 1650": "NVIDIA GTX 1650",
               "RTX 3060": "NVIDIA RTX 3060"}

training.seed_everything(SEEDS[0])

## 1. Data

The patch set of `0_data_preprocess.ipynb`, now with desktop screens in both splits. The
photos and videos are the same files as before, but each was degraded afresh - the codec
differs a little run to run, which shifts every random choice after it - so every model
below, notebook 4's included, is scored again on this set rather than compared with an
old number.

In [ ]:
manifest = training.load_manifest()
training.describe(manifest)
SCALE = manifest["scale"]
train_loader, val_loader = training.loaders(DATA_DIR, manifest, BATCH_SIZE, val_batch_size=128)
trim = training.trimmer(manifest)


def content(source):
    """What a source shows: a drawn desktop screen, a photo (as a moving clip) or a video."""
    if source["dataset"] == "screens":
        return "screen"
    return "photo" if "motion" in source else "video"


for split in ("train", "val"):
    counts = {}
    for source in manifest["splits"][split]["sources"]:
        first, end = source["patches"]
        counts[content(source)] = counts.get(content(source), 0) + end - first
    print(f"{split:<6}" + "  ".join(f"{name} {count:,}" for name, count in counts.items()))

val_sources = manifest["splits"]["val"]["sources"]
shown = [source["patches"][0] for source in val_sources if content(source) == "screen"][::7][:4]
lr = torch.stack([val_loader.dataset[i][0] for i in shown])
hr = torch.stack([val_loader.dataset[i][1] for i in shown])
training.show_patches({"LR (with halo)": lr, "HR": hr})

### How a validation patch is grouped

Every result below is also split by what a patch shows (screen, photo, video), by the
display scale a screen was drawn at (100% is the smallest text), by bitrate and by
downscale filter - and by **what the host sends today**: every ffmpeg line of the host
scales with swscale's bicubic and encodes with x264 `ultrafast`, which has no loop filter.
(Its Windows hardware lines ask `scale_d3d11` for `w=`/`h=`, which the filter calls
`width`/`height`, so a scaled share falls through to that x264 line.)

In [ ]:
screen_info = {r["file"]: r for r in json.loads((sources.RAW / "screens" / "screens.json").read_text())}


def bitrate(degradation):
    bpp = degradation["bpp"]
    if bpp is None:
        return "no codec"
    return "bpp 0.2-0.3" if bpp < 0.3 else "bpp 0.3-0.5" if bpp < 0.5 else "bpp 0.5-1.0"


def labels(source):
    """The groups one source's patches are in."""
    d = source["degradation"]
    out = {"content": content(source), "bitrate": bitrate(d), "filter": d["filter"],
           "host today": "yes" if d["bpp"] is not None and all(d[k] == v for k, v in HOST_TODAY.items()) else "no"}
    if out["content"] == "screen":
        out["display scale"] = f"{screen_info[source['file']]['scale']:.0%}"
    return out


groups = {}   # key -> value -> [validation patch indices]
for source in val_sources:
    first, end = source["patches"]
    for key, value in labels(source).items():
        groups.setdefault(key, {}).setdefault(value, []).extend(range(first, end))
for key, members in groups.items():
    print(f"{key:<14}" + "  ".join(f"{value} {len(idx)}" for value, idx in sorted(members.items())))

### What a model is told

`degrade.condition` turns how a source was made into numbers: for `bpp`, log2 of its
bitrate against the host's default (0.46) and a 1 for no codec; for `path`, the filter
one-hot and a 1 for the loop filter. A told model gets them as a second input, one per
patch, the same for the whole patch.

In [ ]:
for source in val_sources[:3]:
    d = source["degradation"]
    print(f"{d['filter']:<9} bpp {d['bpp'] if d['bpp'] is None else round(d['bpp'], 3)!s:<6} loop filter {d['deblock']!s:<6}"
          + "".join(f"   {name}: {np.round(degrade.condition(d, parts), 2).tolist()}" for name, parts in TOLD.items()))


def loaders_for(parts):
    """The loaders of a model told `parts` (None: the blind loaders)."""
    if parts is None:
        return train_loader, val_loader
    return training.loaders(DATA_DIR, manifest, BATCH_SIZE, val_batch_size=128,
                            condition=lambda source: degrade.condition(source["degradation"], parts))

## 2. Model

Notebook 7's body: a 3x3 stem to 24 channels, two residual blocks, a 3x3 tail that makes
one brightness correction per output pixel, added to R, G and B alike over the bicubic skip.

The told model is the same body with one addition: a 1x1 convolution turns the condition
into a **shift per channel**, added after the stem and inside each residual block, before
its ReLU. It starts at zero, so untrained it is the blind model.

    condition --> 1x1 conv --> + (after the stem)
              --> 1x1 conv --> + (in block 1)      every shift is one number per channel
              --> 1x1 conv --> + (in block 2)

**It ships as the blind graph.** For one stream the condition does not change, so each
shift is a constant per channel, and a constant added after a convolution is that
convolution's bias. `baked(condition)` folds the shifts into the biases and returns a plain
`models.LumaUpscaler`: one input, the client's contract unchanged, the cost unchanged, and
one graph per bitrate (or one per bitrate the client offers) instead of one per model.

In [ ]:
class ToldLuma(models.Upscaler):
    """`models.LumaUpscaler` given a condition (N, K, 1, 1) as a second input, which shifts
    each channel after the stem and inside each residual block."""

    def __init__(self, conditions, width=WIDTH, blocks=BLOCKS, stem=STEM, skip=5, scale=2):
        super().__init__(scale, skip)
        self.width, self.blocks, self.stem = width, blocks, stem
        # the same layers as LumaUpscaler((width,), blocks, stem), in the same order, so the
        # same seed starts both alike and `baked` can copy them across
        self.body = nn.Sequential(nn.Conv2d(3, width, stem, padding=stem // 2), nn.ReLU(inplace=True),
                                  *[models.ResidualBlock(width) for _ in range(blocks)])
        self.tail = models.zero_start(nn.Conv2d(width, scale * scale, 3, padding=1))
        self.halo = self.halo_for(stem, blocks + 1)
        self.shifts = nn.ModuleList([nn.Conv2d(conditions, width, 1, bias=False) for _ in range(blocks + 1)])
        for shift in self.shifts:
            nn.init.zeros_(shift.weight)   # untrained, the condition changes nothing

    def residual(self, x, condition):
        h = torch.relu(self.body[0](x) + self.shifts[0](condition))
        for block, shift in zip(self.body[2:], self.shifts[1:]):
            h = h + block.point(torch.relu(block.conv(h) + shift(condition)))
        luma = self.tail(h)
        return torch.cat([luma, luma, luma], dim=1)

    def forward(self, x, condition):
        return self.shuffle(self.skip(x) + self.residual(x, condition))

    @torch.no_grad()
    def baked(self, condition):
        """The same model for one condition, as a plain LumaUpscaler: each shift folded into
        the bias of the convolution it follows."""
        device = next(self.parameters()).device
        plain = models.LumaUpscaler((self.width,), self.blocks, self.stem, self.skip_size, self.scale).to(device)
        plain.load_state_dict({k: v for k, v in self.state_dict().items() if not k.startswith("shifts.")})
        c = torch.as_tensor(np.asarray(condition), dtype=torch.float32, device=device).reshape(1, -1, 1, 1)
        convs = [plain.body[0], *[block.conv for block in plain.body[2:]]]
        for conv, shift in zip(convs, self.shifts):
            conv.bias += shift(c).flatten()
        return plain.eval()


def build(parts=None):
    """A fresh model: blind (`parts` None), or told those parts of the condition."""
    if parts is None:
        return models.LumaUpscaler((WIDTH,), BLOCKS, STEM, scale=SCALE).to(DEVICE)
    size = sum(degrade.CONDITIONS[part] for part in parts)
    return ToldLuma(size, scale=SCALE).to(DEVICE)


models.check_skip_is_bicubic(SCALE)
blind, told = build(), build(TOLD["told the bitrate and path"])
assert blind.halo == told.halo <= manifest["halo"], "the model reaches further than the patches' halo"
print(f"blind: {models.count_parameters(blind, True):,} learned parameters, told: {models.count_parameters(told, True):,}, "
      f"halo {told.halo}")

# untrained, both are bicubic
probe_lr = next(iter(val_loader))[0][:8].to(DEVICE)
probe_c = torch.randn(8, told.shifts[0].in_channels, 1, 1, device=DEVICE)
with torch.no_grad(), training.full_precision():
    for name, out in (("blind", blind(probe_lr)), ("told", told(probe_lr, probe_c))):
        gap = (trim(out) - trim(models.bicubic(probe_lr, SCALE))).abs().max().item()
        assert gap < 1e-5, f"untrained {name} is not bicubic ({gap:.1e})"

# baking is exact: random weights everywhere, then one condition folded in
check = copy.deepcopy(told)
with torch.no_grad():
    for weight in check.parameters():
        if weight.requires_grad:
            weight.normal_(0, 0.1)
    c = torch.randn(1, check.shifts[0].in_channels, 1, 1, device=DEVICE)
    x = torch.rand(2, 3, 40, 40, device=DEVICE)
    with training.full_precision():
        gap = (check(x, c.expand(2, -1, -1, -1)) - check.baked(c.cpu().numpy())(x)).abs().max().item()
assert gap < 1e-5, f"the baked model computes something else ({gap:.1e})"
print(f"a told model and the plain model baked from it differ by {gap:.1e} (float32 rounding)")

## 3. Speed

What ships is a `models.LumaUpscaler` either way - the blind graph, or one baked from the
told model - so the speed is notebook 7's, counted again here at the client's tile (as in
notebooks 3 and 4: operations and memory traffic in the ONNX graph, the roofline estimate
at the share of its peak this computer's GPU reaches).

In [ ]:
plan = tiles.plan_tiling((FRAME_WIDTH, FRAME_HEIGHT), scale=SCALE, halo=blind.halo)
tile_input = plan["input"][2:]
graph = webexport.export(blind, "speed_check_screens.onnx", tile_input, halo=blind.halo,
                         models_dir=training.CHECKPOINT_DIR / "speed")
assert not graph["outside_budget"], graph["outside_budget"]
work = webexport.count_work(graph["path"])
per_pixel = {"ops": work["operations"] / work["input_pixels"], "bytes": work["activation_bytes"] / work["input_pixels"]}
computed = plan["runs"] * tile_input[0] * tile_input[1]
frame = {"operations": computed * per_pixel["ops"],
         "bytes": computed * per_pixel["bytes"] + plan["runs"] * work["weight_bytes"]}

frame_lr = torch.rand(1, 3, FRAME_HEIGHT // SCALE, FRAME_WIDTH // SCALE, device=DEVICE)
with torch.no_grad():
    measured_ms = speed.time_ms(lambda: blind.eval()(frame_lr), seconds=1.0)
whole = {"operations": frame_lr[0, 0].numel() * per_pixel["ops"],
         "bytes": frame_lr[0, 0].numel() * per_pixel["bytes"] + work["weight_bytes"]}
efficiency = speed.roofline_ms(whole, *speed.HARDWARE[speed.THIS_GPU])[0] / measured_ms if DEVICE.type == "cuda" else 1.0
print(f"{per_pixel['ops']:,.0f} operations and {per_pixel['bytes']:,.0f} bytes per LR pixel; "
      f"{measured_ms:.2f} ms a 1080p frame measured here ({efficiency:.0%} of the peak)")
for short, device in ESTIMATE_ON.items():
    ms, limit = speed.roofline_ms(frame, *speed.HARDWARE[device])
    print(f"  {short:<10}{ms / efficiency:6.1f} ms a 1080p frame ({limit}-bound), {speed.frame_budget(ms / efficiency)}")

## 4. Train

The recipe of notebooks 2, 3, 4 and 7: Adam from 1e-3, batch 32, `metrics.ssim_l1_loss`,
the epoch with the best luma SSIM kept, each run trained until it has converged. The blind
model three times (seeds 0, 1, 2); each told model once, at seed 0.

**A told model and the blind one at the same seed differ in one thing only.** They start
from the same weights - the shifts start at zero - and see the same batches in the same
order, so the gap between them is what the condition buys, with none of a re-draw's luck
in it. The three blind seeds still say how large luck is.

Each run saves its training to `checkpoints/screens_runs/` after every epoch: run the cell
again after an interruption and it carries on; finished runs just load.

In [ ]:
def train(name, parts=None, seed=0):
    """Train a fresh model, or load its saved training. Returns it at its best epoch,
    its history and its best epoch."""
    torch.manual_seed(seed)
    model = build(parts)
    torch.manual_seed(seed)   # the same batches in the same order, told or not
    train_set, val_set = loaders_for(parts)
    run = "_".join(name.split()) + f"_seed{seed}"
    history, best = training.train(model, train_set, val_set, trim, MAX_EPOCHS, learning_rate=LEARNING_RATE,
                                   loss_fn=metrics.ssim_l1_loss, select="ssim", checkpoint=RUN_DIR / f"{run}.pt",
                                   verbose=0)
    print(f"{name:<28} seed {seed}: best epoch {best['epoch']:3} of {len(history):3}, ssim-y {best['score']:.4f}")
    return model, history, best


trained, histories, bests = {}, {}, {}
for seed in SEEDS:
    name = f"blind, seed {seed}"
    trained[name], histories[name], bests[name] = train("blind", seed=seed)
for name, parts in TOLD.items():
    trained[name], histories[name], bests[name] = train(name, parts, SEEDS[0])

# notebook 4's winner: the same body, trained on the set before it had screens
before = training.load_checkpoint(BEFORE)
assert before["architecture"] == "LumaUpscaler", before["architecture"]
trained["before screens"] = models.LumaUpscaler(scale=SCALE, **before["arguments"]).to(DEVICE)
trained["before screens"].load_state_dict(before["state_dict"])
PARTS = {"before screens": None, **{f"blind, seed {seed}": None for seed in SEEDS}, **TOLD}

## 5. Results

### Overall

Every model on the whole validation set, against the mean of the blind model's three
seeds. **better** / **worse** means the gap is wider than the seeds' spread; **luck** means
this set cannot tell them apart.

In [ ]:
val_sets = {parts: loaders_for(parts)[1].dataset for parts in {None, *TOLD.values()}}


def score(name, indices=None):
    """`metrics.report` of one model (or "bicubic") on the validation patches `indices` (None: all)."""
    parts = PARTS.get(name)
    dataset = val_sets[parts]
    if indices is not None:
        dataset = Subset(dataset, indices)
    model = models.Resize(SCALE).to(DEVICE) if name == "bicubic" else trained[name]
    return training.evaluate(model, DataLoader(dataset, batch_size=128), trim)


results = {name: score(name) for name in ["bicubic", *trained]}
seeds = [results[f"blind, seed {seed}"] for seed in SEEDS]
spread = {key: max(s[key] for s in seeds) - min(s[key] for s in seeds) for key in ("ssim", "wpsnr")}
blind_mean = {key: float(np.mean([s[key] for s in seeds])) for key in ("ssim", "wpsnr")}


def verdict(name, scores):
    if name == "bicubic" or name.startswith("blind"):
        return ""
    gap = scores["ssim"] - blind_mean["ssim"]
    word = "luck" if abs(gap) <= spread["ssim"] else ("better" if gap > 0 else "worse")
    return f"{gap:+11.4f}  {word}"


print(f"the blind model at {len(SEEDS)} seeds: {spread['ssim']:.4f} ssim-y and {spread['wpsnr']:.2f} dB wpsnr apart; "
      f"last column: ssim-y against their mean of {blind_mean['ssim']:.4f}")
metrics.print_table(results, extra=verdict)

### By group

The gain over bicubic in each group: luma SSIM, and luma PSNR in dB. `blind` is the mean
of the three seeds, and its `±` is their spread in that group - the ruler for the columns
beside it.

In [ ]:
SHOW = ["before screens", "blind", *TOLD]


def group_table(key):
    print(f"\n{key}: gain over bicubic, ssim-y / psnr-y dB")
    print(f"{'':<16}{'patches':>8}{'bicubic ssim-y':>16}" + "".join(f"{name:>30}" for name in SHOW))
    for value, indices in sorted(groups[key].items()):
        base = score("bicubic", indices)
        row = f"{value:<16}{len(indices):8}{base['ssim']:16.4f}"
        for name in SHOW:
            if name == "blind":
                runs = [score(f"blind, seed {seed}", indices) for seed in SEEDS]
                ssim = [r["ssim"] - base["ssim"] for r in runs]
                y = [r["y"] - base["y"] for r in runs]
                cell = f"{np.mean(ssim):+.4f}±{max(ssim) - min(ssim):.4f} / {np.mean(y):+.2f}"
            else:
                r = score(name, indices)
                cell = f"{r['ssim'] - base['ssim']:+.4f} / {r['y'] - base['y']:+.2f}"
            row += f"{cell:>30}"
        print(row)


for key in ("content", "display scale", "bitrate", "filter", "host today"):
    group_table(key)

### What it looks like

A held-out screen, through the stream the way the host sends it today (swscale's bicubic,
x264 `ultrafast`, no loop filter, the host's default 0.46 bpp, a still screen after a
second), then upscaled whole: bicubic, the model trained before there were screens, and the
best one now.

In [ ]:
order = sorted([name for name in trained if name != "before screens"], key=lambda name: -results[name]["ssim"])
best_name = order[0]

# a held-out screen at 100% or 125% (the smallest text), with code on it, or a document
pick = min((r for r in screen_info.values() if r["split"] == "test" and r["scale"] <= 1.25),
           key=lambda r: {"code": 0, "document": 1}.get(r["apps"][0], 2))
picture = np.asarray(Image.open(sources.RAW / "screens" / pick["file"]).convert("RGB"))[:720, :1280]
host = degrade.Degradation(scale=SCALE, filter=HOST_TODAY["filter"], bpp=degrade.STREAM_BPP,
                           deblock=HOST_TODAY["deblock"])
truth, lr_clip = degrade.degrade(np.repeat(picture[None], 30, axis=0), host)
target, lr = training.to_tensor(truth[-1]), training.to_tensor(lr_clip[-1])


@torch.no_grad()
def upscale(name, lr):
    """A whole frame through a model, its edge padded by repetition like a tiled client."""
    if name == "bicubic":
        return models.bicubic(lr, SCALE).clamp(0, 1)
    halo = blind.halo
    padded = F.pad(lr, (halo, halo, halo, halo), mode="replicate")
    model = trained[name]
    if PARTS.get(name) is not None:
        model = model.baked(degrade.condition(host, PARTS[name]))
    return patches.trim(model.eval()(padded), halo * SCALE).clamp(0, 1)


pictures = {name: upscale(name, lr) for name in dict.fromkeys(["bicubic", "before screens", best_name])}
metrics.print_table({name: metrics.report(image, target) for name, image in pictures.items()})
Image.fromarray(training.to_image(pictures[best_name])).save(training.SAMPLE_DIR / "upscaled_screens.png")

# show the busiest 200x360 of it: the most edges, which on a screen is the most text
edges = np.abs(np.diff(training.to_image(target).astype(np.float32).mean(axis=2), axis=1))
top, left = max(((y, x) for y in range(0, edges.shape[0] - 200, 20) for x in range(0, edges.shape[1] - 360, 20)),
                key=lambda p: edges[p[0]:p[0] + 200, p[1]:p[1] + 360].mean())
box = (slice(top, top + 200), slice(left, left + 360))
panels = {**pictures, "target": target}
fig, axes = plt.subplots(1, len(panels), figsize=(4.6 * len(panels), 3))
for ax, (name, image) in zip(axes, panels.items()):
    ax.imshow(training.to_image(image)[box], interpolation="nearest")
    ax.set_title(name, fontsize=10)
    ax.axis("off")
fig.tight_layout()
plt.show()
print(f"{pick['file']}: {pick['apps']}, {pick['scale']:.0%} display scale, {pick['theme']} theme")

## 6. Save, reload and export

The best model by luma SSIM is saved as it ships. A told model is saved whole, with what it
is told, and exported **baked**: one plain graph per bitrate in `BAKE_BPP`, each for the
host's path today (bicubic, no loop filter) - single input, the client's contract unchanged.
At the client's tile (a 320x180 step), float32 and float16, and float16 as a whole 1080p
frame at the host's default bitrate.

In [ ]:
best = trained[best_name]
parts = PARTS.get(best_name)
if parts is None:
    arguments = {"widths": [WIDTH], "blocks": BLOCKS, "stem": STEM}
else:
    arguments = {"conditions": best.shifts[0].in_channels, "width": WIDTH, "blocks": BLOCKS, "stem": STEM}
training.save_checkpoint(CHECKPOINT, best, architecture=type(best).__name__, arguments=arguments, told=list(parts or []),
                         scale=SCALE, halo=best.halo, loss="ssim_l1_loss", select="ssim",
                         epochs=len(histories[best_name]), best_epoch=bests[best_name]["epoch"],
                         val=results[best_name], val_bicubic=results["bicubic"], blind_mean=blind_mean,
                         seed_spread=spread, host_today=HOST_TODAY, degradation=manifest["degradation"])

# rebuild it from the file alone, the way a later notebook would
checkpoint = training.load_checkpoint(CHECKPOINT)
if checkpoint["told"]:
    reloaded = ToldLuma(scale=checkpoint["scale"], **checkpoint["arguments"]).to(DEVICE)
    reloaded.load_state_dict(checkpoint["state_dict"])
    told_today = degrade.condition(host, tuple(checkpoint["told"]))
    training.assert_same_model(best.baked(told_today), reloaded.baked(told_today))
else:
    reloaded = models.LumaUpscaler(scale=checkpoint["scale"], **checkpoint["arguments"]).to(DEVICE)
    reloaded.load_state_dict(checkpoint["state_dict"])
    training.assert_same_model(best, reloaded)
print("reloaded model is identical")


def input_shape(step_height, step_width, halo):
    return (step_height + 2 * halo, step_width + 2 * halo)


def check_and_time(info, module, shape, dtype, tolerance):
    webexport.check_graph(info["path"], module, np.random.rand(1, 3, *shape).astype(np.float32), dtype, tolerance)
    ms = webexport.time_graph(info["path"], shape, dtype)
    webexport.annotate(info, cpu_ms=round(ms, 2))
    print(webexport.summary_line(info, ms))


tile_name = f"tile{STEP_WIDTH}x{STEP_HEIGHT}"
halo = best.halo
tile_shape = input_shape(STEP_HEIGHT, STEP_WIDTH, halo)
frame_shape = input_shape(FRAME_HEIGHT // SCALE, FRAME_WIDTH // SCALE, halo)
variants = {"": best} if parts is None else {
    f"_bpp{bpp:g}": best.baked(degrade.condition(degrade.Degradation(scale=SCALE, bpp=bpp, **HOST_TODAY), parts))
    for bpp in BAKE_BPP}
for suffix, module in variants.items():
    label = f"{NAME}{suffix.replace('_', ' ')}"
    fp32 = webexport.export(module, f"upscale_{NAME}{suffix}_{tile_name}.onnx", tile_shape, label=f"{label} fp32",
                            halo=halo)
    check_and_time(fp32, module, tile_shape, np.float32, tolerance=2e-3)
    fp16 = webexport.to_float16(fp32, f"upscale_{NAME}{suffix}_fp16_{tile_name}.onnx")
    webexport.write_metadata(fp16["path"], {"label": f"{label} fp16"})
    check_and_time(fp16, module, tile_shape, np.float16, tolerance=4e-3)
    if suffix in ("", f"_bpp{degrade.STREAM_BPP:g}"):
        frame = webexport.to_frame(fp16, f"upscale_{NAME}{suffix}_fp16_{tile_name}_frame.onnx", FRAME_WIDTH, FRAME_HEIGHT)
        check_and_time(frame, module, frame_shape, np.float16, tolerance=4e-3)
print("compare in a browser: uv run src/upscale/benchmark/main.py")

## What to look at

- **Screens against photos and video.** How much better a model trained with screens is on
  them than one that never saw a glyph - and whether it paid for that on photos and video.
- **Display scale.** 100% is the smallest text: the hardest case, and the commonest on a
  1080p desktop streamed at half its height.
- **Told against blind, at the same seed.** The only difference between the two is the
  condition, so their gap is what knowing the stream is worth - bitrate alone (what the
  client always knows), then the encoder path too (what it knows if the host says).
- **The host today.** Whatever wins overall has to win on that row, because it is what a
  client receives.